# ⏱ TimesFM 완전 분석
## Google Research · Decoder-only 시계열 파운데이션 모델

> **논문**: "A decoder-only foundation model for time-series forecasting" (Das et al., ICML 2024)  
> **GitHub**: https://github.com/google-research/timesfm  
> **HuggingFace**: google/timesfm-2.5-200m-pytorch

---
### 📚 목차
1. [환경 설정 & 데이터 준비](#1)
2. [핵심 개념: 아키텍처 다이어그램](#2)
3. [사전학습 데이터 구성](#3)
4. [버전 진화 비교](#4)
5. [벤치마크 1: Monash 아카이브 성능](#5)
6. [벤치마크 2: GIFT-Eval 2026 현황](#6)
7. [스케일링 법칙](#7)
8. [활용 분야 적합도 분석](#8)
9. [한계 레이더 차트](#9)
10. [모델 선택 가이드 요약](#10)


## 1. 환경 설정 & 데이터 준비 <a id='1'></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import warnings
warnings.filterwarnings('ignore')

# ── 스타일 설정 ─────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#8b949e',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'text.color':       '#e6edf3',
    'grid.color':       '#21262d',
    'grid.alpha':       0.6,
    'font.family':      'DejaVu Sans',
    'font.size':        11,
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
    'axes.titlecolor':  '#e6edf3',
})

BLUE   = '#4285f4'
PURPLE = '#6366f1'
GREEN  = '#34d399'
ORANGE = '#f59e0b'
RED    = '#ef4444'
GRAY   = '#64748b'
PINK   = '#ec4899'
CYAN   = '#06b6d4'

print("✅ 환경 설정 완료!")
print(f"   matplotlib: {plt.matplotlib.__version__}")
print(f"   numpy:      {np.__version__}")
print(f"   pandas:     {pd.__version__}")


: 

## 2. 핵심 개념: 아키텍처 다이어그램 <a id='2'></a>

TimesFM은 **GPT와 동일한 Decoder-only Transformer** 구조를 시계열에 적용합니다.  
핵심 차이는 텍스트 토큰 대신 **숫자 값의 묶음(패치)**을 토큰으로 사용한다는 점입니다.

### 핵심 수식
- **입력 패치**: 연속 32개 값 → 1개 토큰
- **출력 패치**: 1개 토큰 → 미래 128개 값 예측  
- **학습 목적함수**: Next-patch MSE (Teacher-forcing)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.patch.set_facecolor('#0d1117')

# ── 왼쪽: 데이터 흐름 다이어그램 ──────────────────────────
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)
ax.axis('off')
ax.set_facecolor('#0d1117')
ax.set_title('TimesFM 데이터 흐름', pad=15)

steps = [
    (0.3, 3.0, '📈 원본\n시계열',     '임의 길이 입력',          BLUE),
    (2.2, 3.0, '✂️ 패치\n분할',       '32개 값씩 자름',           PURPLE),
    (4.1, 3.0, '🔢 패치\n임베딩',     'MLP → 1280dim',           '#8b5cf6'),
    (6.0, 3.0, '🧠 인과\n어텐션',     '16 heads, 20 layers',      '#a78bfa'),
    (7.9, 3.0, '🎯 미래\n예측',       '128값 출력',               GREEN),
]

for x, y, title, sub, color in steps:
    rect = FancyBboxPatch((x-0.75, y-0.85), 1.5, 1.7,
                           boxstyle="round,pad=0.1",
                           facecolor=color+'28', edgecolor=color, linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x, y+0.3, title, ha='center', va='center',
            fontsize=10, fontweight='bold', color=color)
    ax.text(x, y-0.45, sub, ha='center', va='center',
            fontsize=8, color='#8b949e')

# 화살표
for i in range(len(steps)-1):
    x1 = steps[i][0] + 0.75
    x2 = steps[i+1][0] - 0.75
    y  = steps[i][1]
    ax.annotate('', xy=(x2, y), xytext=(x1, y),
                arrowprops=dict(arrowstyle='->', color='#30363d', lw=2))

# 입력/출력 패치 크기 강조
ax.annotate('', xy=(3.35, 1.5), xytext=(1.45, 1.5),
            arrowprops=dict(arrowstyle='<->', color=BLUE, lw=2))
ax.text(2.4, 1.2, '입력 패치 = 32', ha='center', fontsize=9, color=BLUE, fontweight='bold')

ax.annotate('', xy=(8.65, 1.5), xytext=(7.15, 1.5),
            arrowprops=dict(arrowstyle='<->', color=GREEN, lw=2))
ax.text(7.9, 1.2, '출력 패치 = 128', ha='center', fontsize=9, color=GREEN, fontweight='bold')

ax.text(5.0, 0.4, '⚡ 출력(128) > 입력(32) → 512스텝 예측 시 자기회귀 4회만 필요 → 오차 누적 최소화',
        ha='center', fontsize=9, color=ORANGE,
        bbox=dict(boxstyle='round', facecolor=ORANGE+'18', edgecolor=ORANGE+'60'))

# ── 오른쪽: 어텐션 패턴 시각화 ────────────────────────────
ax2 = axes[1]
ax2.set_facecolor('#161b22')
ax2.set_title('인과 어텐션 마스크 (Causal Mask)
과거 패치만 참조 가능 — 미래 누설 방지', pad=15)

n = 6
mask = np.tril(np.ones((n, n)))
attn = mask * np.random.uniform(0.3, 1.0, (n, n))
for i in range(n):
    row_sum = attn[i].sum()
    if row_sum > 0:
        attn[i] /= row_sum

im = ax2.imshow(attn, cmap='Blues', vmin=0, vmax=1, aspect='auto')
labels = [f'패치 {i+1}' for i in range(n)]
ax2.set_xticks(range(n)); ax2.set_xticklabels(labels, fontsize=9)
ax2.set_yticks(range(n)); ax2.set_yticklabels(labels, fontsize=9)
ax2.set_xlabel('Key (참조 대상)', color='#8b949e')
ax2.set_ylabel('Query (현재 위치)', color='#8b949e')

for i in range(n):
    for j in range(n):
        val = attn[i, j]
        color_txt = 'white' if val > 0.4 else '#8b949e'
        ax2.text(j, i, f'{val:.2f}' if mask[i,j] else '✗',
                ha='center', va='center', fontsize=9, color=color_txt)

cb = plt.colorbar(im, ax=ax2, shrink=0.8)
cb.ax.yaxis.set_tick_params(color='#8b949e')
plt.setp(cb.ax.yaxis.get_ticklabels(), color='#8b949e')

plt.tight_layout(pad=2)
plt.show()
print("💡 핵심: 각 패치는 자신보다 이전 패치들만 참조 (인과성 보장)")


## 3. 사전학습 데이터 구성 <a id='3'></a>

TimesFM은 **100B+ 타임포인트**로 사전학습 되었으며,  
데이터 구성이 모델의 강점과 약점을 결정합니다.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0d1117')

# ── 파이 차트 ──────────────────────────────────────────────
ax1 = axes[0]
ax1.set_facecolor('#0d1117')

labels  = ['Wikipedia\n페이지뷰', '합성 시계열\n(ARMA+Sin)', '소매/판매\n(Favorita)',
           '교통량\n(LibCity)', '전력 소비', 'Google Trends', '기타']
sizes   = [374, 61, 14, 3.4, 0.8, 0.5, 1.3]
colors  = [BLUE, PURPLE, '#8b5cf6', '#a78bfa', CYAN, '#06b6d4', GRAY]
explode = [0.05, 0.02, 0, 0, 0, 0, 0]

wedges, texts, autotexts = ax1.pie(
    sizes, labels=labels, colors=colors, explode=explode,
    autopct='%1.1f%%', startangle=140,
    textprops={'color': '#e6edf3', 'fontsize': 9},
    pctdistance=0.75, wedgeprops={'linewidth': 1.5, 'edgecolor': '#0d1117'}
)
for at in autotexts:
    at.set_fontsize(8)
    at.set_color('white')
    at.set_fontweight('bold')

ax1.set_title('사전학습 데이터 비율
(단위: 십억 타임포인트)', pad=15)
ax1.text(0, -1.45,
         '⚠️ Wikipedia가 80%+ → 웹 트래픽 계열에서 특히 강점',
         ha='center', fontsize=9, color=ORANGE,
         bbox=dict(boxstyle='round', facecolor=ORANGE+'18', edgecolor=ORANGE+'40'))

# ── 학습 데이터 총량 비교 ──────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor('#161b22')

models  = ['GIFT-Eval\nPretrain', 'TimesFM\n1.0/2.0', 'TimeGPT-1\n(Nixtla)',
           'Chronos\n(Amazon)', 'Moirai\nLOTSA', 'MOMENT\n(CMU)']
points  = [240, 100, 100, 84, 27, 1.13]
clrs    = [ORANGE, BLUE, CYAN, '#ff9900', '#00a1e0', GRAY]

bars = ax2.barh(models, points, color=clrs, edgecolor='#30363d', height=0.55)
ax2.set_xlabel('사전학습 데이터 크기 (십억 타임포인트)', color='#8b949e')
ax2.set_title('파운데이션 모델별 학습 데이터 규모', pad=15)
ax2.grid(axis='x', alpha=0.4)
ax2.set_xscale('log')

for bar, val in zip(bars, points):
    ax2.text(val * 1.15, bar.get_y() + bar.get_height()/2,
             f'{val}B', va='center', fontsize=10, fontweight='bold',
             color='#e6edf3')

plt.tight_layout(pad=2)
plt.show()


## 4. 버전 진화 비교 <a id='4'></a>

v1.0 → v2.0 → v2.5: **더 작게, 더 길게, 더 정확하게**  
v2.5의 역설: 파라미터를 절반으로 줄였지만 성능은 더 좋아짐 → **데이터 다양성 > 파라미터 수**


In [ ]:
fig = plt.figure(figsize=(18, 8))
fig.patch.set_facecolor('#0d1117')
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

versions  = ['v1.0\n(2024.02)', 'v2.0\n(2024.12)', 'v2.5\n(2025.09)']
params    = [200, 500, 200]
context   = [512, 2048, 16384]
layers    = [20, 50, 20]
mase      = [1.08, 0.76, 0.71]
clrs      = [BLUE, PURPLE, GREEN]

specs = [
    (gs[0,0], params,   '파라미터 수 (M)',             '빌드 비용'),
    (gs[0,1], context,  '최대 컨텍스트 길이 (토큰)',    '참조 가능한 과거 길이'),
    (gs[0,2], layers,   'Transformer 레이어 수',        '모델 깊이'),
]
for gspec, vals, title, sub in specs:
    ax = fig.add_subplot(gspec)
    ax.set_facecolor('#161b22')
    bars = ax.bar(versions, vals, color=clrs, edgecolor='#30363d', width=0.5)
    ax.set_title(f'{title}\n{sub}', fontsize=11)
    ax.grid(axis='y', alpha=0.4)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.03,
                f'{val:,}', ha='center', fontsize=11, fontweight='bold', color='#e6edf3')

# MASE 꺾은선
ax_mase = fig.add_subplot(gs[1, :2])
ax_mase.set_facecolor('#161b22')
ax_mase.plot([0,1,2], mase, 'o-', color=GREEN, linewidth=2.5, markersize=10, markerfacecolor='#0d1117', markeredgewidth=2.5)
for i, (v, m) in enumerate(zip(versions, mase)):
    ax_mase.annotate(f'MASE {m}', xy=(i, m), xytext=(i, m+0.04),
                     ha='center', fontsize=11, fontweight='bold', color=clrs[i])
ax_mase.axhline(1.0, color=RED, linestyle='--', alpha=0.6, linewidth=1.5, label='Seasonal Naive (MASE=1.00)')
ax_mase.set_xticks([0,1,2]); ax_mase.set_xticklabels(versions)
ax_mase.set_ylim(0.6, 1.2)
ax_mase.set_title('GIFT-Eval MASE 개선 추이 (낮을수록 좋음)', fontsize=11)
ax_mase.set_ylabel('MASE')
ax_mase.legend(loc='upper right', facecolor='#1c2128', edgecolor='#30363d')
ax_mase.grid(alpha=0.4)

# 변경사항 요약 텍스트
ax_note = fig.add_subplot(gs[1, 2])
ax_note.set_facecolor('#0d1117')
ax_note.axis('off')
changes = [
    ('v1.0 → v2.0', [
        '• 파라미터 200M → 500M',
        '• Sinusoidal PE → RoPE',
        '• 컨텍스트 512 → 2048',
        '• LOTSA 데이터 추가',
    ], PURPLE),
    ('v2.0 → v2.5', [
        '• 파라미터 500M → 200M ↓',
        '• 컨텍스트 2048 → 16384 ↑↑',
        '• 주파수 토큰 제거',
        '• 보정된 분위수 헤드 추가',
        '• RevIN 정규화 도입',
    ], GREEN),
]
y = 0.95
for title, items, color in changes:
    ax_note.text(0.05, y, title, fontsize=11, fontweight='bold', color=color, transform=ax_note.transAxes)
    y -= 0.08
    for item in items:
        ax_note.text(0.05, y, item, fontsize=9, color='#8b949e', transform=ax_note.transAxes)
        y -= 0.07
    y -= 0.04

plt.suptitle('TimesFM 버전 진화: v1.0 → v2.0 → v2.5', fontsize=15, fontweight='bold', color='#e6edf3', y=1.01)
plt.show()
print("💡 핵심 교훈: v2.5는 v2.0의 절반 파라미터로 더 높은 성능 → 데이터 품질이 규모보다 중요!")


## 5. 벤치마크 1: Monash 아카이브 성능 <a id='5'></a>

TimesFM 논문(ICML 2024)의 핵심 결과: **Zero-shot으로 지도학습 모델과 동등**
- Naive 기준(1.00) 대비 낮을수록 좋음
- TimesFM-ZS: 추가 학습 없이 가장 좋은 성능


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.patch.set_facecolor('#0d1117')

# ── 왼쪽: Monash GM-MASE 가로 막대 ───────────────────────
ax = axes[0]
ax.set_facecolor('#161b22')

data = [
    ('TimesFM-ZS ★',  0.6846, 'foundation'),
    ('N-BEATS',        0.7005, 'supervised'),
    ('FFNN',           0.7044, 'supervised'),
    ('DeepAR',         0.7477, 'supervised'),
    ('CatBoost',       0.7733, 'supervised'),
    ('TBATS',          0.7736, 'statistical'),
    ('ETS',            0.8104, 'statistical'),
    ('WaveNet',        0.9384, 'supervised'),
    ('ARIMA',          0.9449, 'statistical'),
    ('llmtime-ZS',     0.9715, 'llm'),
    ('Naive',          1.0000, 'baseline'),
    ('PatchTST-ZS',    1.0557, 'foundation'),
]
data_sorted = sorted(data, key=lambda x: x[1])

type_colors = {
    'foundation': GREEN, 'supervised': PURPLE,
    'statistical': ORANGE, 'llm': PINK, 'baseline': GRAY
}
type_labels = {
    'foundation': '파운데이션 모델 (ZS)',
    'supervised': '지도학습 딥러닝',
    'statistical': '통계 모델',
    'llm': 'LLM 방식',
    'baseline': 'Baseline'
}

models = [d[0] for d in data_sorted]
scores = [d[1] for d in data_sorted]
bar_colors = [
    (GREEN if d[0].startswith('TimesFM') else RED if d[0].startswith('PatchTST') else type_colors[d[2]])
    for d in data_sorted
]

bars = ax.barh(models, scores, color=bar_colors, edgecolor='#30363d', height=0.65)
ax.axvline(1.0, color=RED, linestyle='--', alpha=0.7, linewidth=1.5, label='Naive 기준 (1.00)')
ax.set_xlabel('GM-Scaled MAE (낮을수록 좋음)')
ax.set_title('Monash 아카이브 Zero-shot 성능
(18개 데이터셋 기하평균)', pad=12)
ax.grid(axis='x', alpha=0.4)
ax.set_xlim(0.5, 1.15)

for bar, score, d in zip(bars, scores, data_sorted):
    ax.text(score + 0.005, bar.get_y() + bar.get_height()/2,
            f'{score:.4f}', va='center', fontsize=9, color='#e6edf3')

# 범례
seen = set()
handles = []
for d in data_sorted:
    t = d[2]
    if t not in seen:
        c = GREEN if t == 'foundation' else type_colors[t]
        handles.append(mpatches.Patch(color=c, label=type_labels[t]))
        seen.add(t)
handles.append(plt.Line2D([0],[0], color=RED, linestyle='--', label='Naive 기준'))
ax.legend(handles=handles, loc='lower right', facecolor='#1c2128', edgecolor='#30363d', fontsize=9)

# ── 오른쪽: 개선율 비교 ──────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor('#161b22')

comparisons = [
    ('vs. Naive 기준',       1.00,   0.6846),
    ('vs. ARIMA',            0.9449, 0.6846),
    ('vs. llmtime (GPT-3)',  0.9715, 0.6846),
    ('vs. PatchTST-ZS',      1.0557, 0.6846),
    ('vs. DeepAR (지도학습)', 0.7477, 0.6846),
    ('vs. N-BEATS (지도학습)',0.7005, 0.6846),
]
comp_labels = [c[0] for c in comparisons]
improvements = [round((c[1] - c[2]) / c[1] * 100, 1) for c in comparisons]
bar_clrs = [GREEN if imp > 0 else RED for imp in improvements]

bars2 = ax2.barh(comp_labels, improvements, color=bar_clrs, edgecolor='#30363d', height=0.55)
ax2.axvline(0, color='#30363d', linewidth=1)
ax2.set_xlabel('TimesFM 개선율 (%)')
ax2.set_title('TimesFM-ZS 상대 개선율
(양수 = TimesFM이 더 좋음)', pad=12)
ax2.grid(axis='x', alpha=0.4)

for bar, val in zip(bars2, improvements):
    x = val + 0.3 if val >= 0 else val - 0.3
    ha = 'left' if val >= 0 else 'right'
    ax2.text(x, bar.get_y() + bar.get_height()/2,
             f'+{val}%' if val >= 0 else f'{val}%',
             va='center', fontsize=10, fontweight='bold',
             color=GREEN if val >= 0 else RED)

plt.tight_layout(pad=2)
plt.show()
print("✅ TimesFM Zero-shot은 지도학습 N-BEATS와 동등하고, llmtime(GPT-3)보다 29% 우수!")


## 6. 벤치마크 2: GIFT-Eval 2026 현황 <a id='6'></a>

**GIFT-Eval**: Salesforce가 관리하는 97개 데이터셋 리더보드 (2026년 4월 기준)  
비누출(non-leaking) 조건: 사전학습 데이터에 포함되지 않은 데이터셋만 평가


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.patch.set_facecolor('#0d1117')

# ── GIFT-Eval MASE 랭킹 ──────────────────────────────────
ax = axes[0]
ax.set_facecolor('#161b22')

gift_data = [
    ('Chronos-2 (Amazon)',     0.70, '#ff9900', False),
    ('TiRex (NXAI)',           0.72, '#8b5cf6', False),
    ('TimesFM-2.5 ★ (Google)', 0.71, GREEN,     False),
    ('Moirai-2 (Salesforce)',  0.73, '#00a1e0', False),
    ('Toto-Open (Datadog)',    0.75, PURPLE,    False),
    ('TimesFM-2.0 (Google)',   0.76, BLUE,      True),
    ('PatchTST (Princeton)',   0.85, CYAN,      False),
    ('Chronos-Large (Amazon)', 0.87, '#ff9900', True),
    ('Auto-ARIMA',             1.07, GRAY,      False),
    ('TimesFM-1.0 (Google)',   1.08, '#4285f4', True),
    ('Seasonal-Naive',         1.00, '#30363d', False),
]
gift_sorted = sorted(gift_data, key=lambda x: x[1])

g_models = [d[0] for d in gift_sorted]
g_scores = [d[1] for d in gift_sorted]
g_colors = [d[2] for d in gift_sorted]
g_leaked = [d[3] for d in gift_sorted]

bars = ax.barh(g_models, g_scores, color=g_colors, edgecolor='#30363d', height=0.65,
               alpha=[0.5 if lk else 1.0 for lk in g_leaked])

# 누출 표시
for bar, lk in zip(bars, g_leaked):
    if lk:
        ax.text(0.51, bar.get_y() + bar.get_height()/2,
                '⚠️ 데이터 누출', va='center', fontsize=8, color=ORANGE, style='italic')

ax.axvline(1.0, color=RED, linestyle='--', alpha=0.6, linewidth=1.5)
ax.set_xlabel('MASE (낮을수록 좋음)')
ax.set_title('GIFT-Eval 랭킹 (2026.04 기준)
흐린 막대 = 사전학습 데이터 누출 의심', pad=12)
ax.grid(axis='x', alpha=0.4)
ax.set_xlim(0.5, 1.2)

for bar, score in zip(bars, g_scores):
    ax.text(score + 0.005, bar.get_y() + bar.get_height()/2,
            f'{score:.2f}', va='center', fontsize=9, color='#e6edf3')

# ── 파라미터 vs MASE 산점도 ──────────────────────────────
ax2 = axes[1]
ax2.set_facecolor('#161b22')

scatter_data = [
    ('TimesFM-2.5', 200, 0.71, GREEN,    'Google'),
    ('TimesFM-2.0', 500, 0.76, BLUE,     'Google'),
    ('TimesFM-1.0', 200, 1.08, '#4285f4','Google'),
    ('Chronos-2',   120, 0.70, '#ff9900','Amazon'),
    ('Chronos-Large',710,0.87, '#ff9900','Amazon'),
    ('Moirai-2',    14,  0.73, '#00a1e0','Salesforce'),
    ('TiRex',       35,  0.72, '#8b5cf6','NXAI'),
    ('Toto-Open',   151, 0.75, PURPLE,   'Datadog'),
    ('PatchTST',    10,  0.85, CYAN,     'Princeton'),
    ('Auto-ARIMA',  0.1, 1.07, GRAY,     'Classical'),
]

for name, params, mase_v, color, org in scatter_data:
    ax2.scatter(params, mase_v, s=120, color=color, edgecolors='#30363d',
                linewidth=1.5, zorder=5)
    offset_x = 15
    offset_y = 0.01
    if name == 'TimesFM-1.0': offset_y = -0.03
    if name == 'Chronos-2':   offset_x = -70
    ax2.annotate(name, (params, mase_v), textcoords='offset points',
                 xytext=(offset_x, 5), fontsize=8, color=color)

ax2.axhline(1.0, color=RED, linestyle='--', alpha=0.5, linewidth=1.2, label='Naive 기준')
ax2.set_xlabel('파라미터 수 (M)')
ax2.set_ylabel('GIFT-Eval MASE')
ax2.set_title('파라미터 수 vs 예측 정확도
(오른쪽 아래가 이상적)', pad=12)
ax2.grid(alpha=0.4)
ax2.legend(facecolor='#1c2128', edgecolor='#30363d')

ax2.text(300, 1.05,
         '크다고 좋은 게 아님!
Moirai-2(14M)이
Chronos-Large(710M)보다 우수',
         fontsize=9, color=ORANGE, ha='center',
         bbox=dict(boxstyle='round', facecolor=ORANGE+'18', edgecolor=ORANGE+'40'))

plt.tight_layout(pad=2)
plt.show()
print("⚠️ 2026 현실: Chronos-2가 GIFT-Eval 1위. TimesFM-2.5는 Top 3 수준.")
print("   그러나 BigQuery SQL 통합 및 오픈소스 측면에서 TimesFM은 여전히 1위!")


## 7. 스케일링 법칙 & 패치 크기 Ablation <a id='7'></a>

논문 Figure 3에서 발췌한 핵심 실험 결과


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0d1117')

# 스케일링
ax1 = axes[0]
ax1.set_facecolor('#161b22')
params_list = [17, 70, 200]
mase_list   = [0.76, 0.71, 0.6846]
ax1.plot(params_list, mase_list, 'o-', color=PURPLE, linewidth=2.5,
         markersize=10, markerfacecolor='#0d1117', markeredgewidth=2.5)
for p, m in zip(params_list, mase_list):
    ax1.annotate(f'{p}M params
MASE={m}', (p, m),
                 textcoords='offset points', xytext=(10, 5),
                 fontsize=10, color=PURPLE, fontweight='bold')
ax1.set_xlabel('파라미터 수 (M)'); ax1.set_ylabel('Monash GM-MASE')
ax1.set_title('스케일링 법칙
파라미터가 늘수록 성능 향상 (단조 감소)', pad=12)
ax1.grid(alpha=0.4)
ax1.set_ylim(0.65, 0.80)
ax1.text(100, 0.775, '하지만 v2.5는
데이터 다양성으로
200M에서도 0.71 달성!',
         fontsize=9, color=GREEN,
         bbox=dict(boxstyle='round', facecolor=GREEN+'18', edgecolor=GREEN+'40'))

# 패치 크기 Ablation
ax2 = axes[1]
ax2.set_facecolor('#161b22')
patch_sizes = [8, 16, 32, 64, 128]
ett_mae     = [0.42, 0.39, 0.36, 0.38, 0.41]
bar_clrs    = [GREEN if s == 32 else PURPLE for s in patch_sizes]
bars = ax2.bar([str(s) for s in patch_sizes], ett_mae, color=bar_clrs, edgecolor='#30363d', width=0.55)
ax2.set_xlabel('입력 패치 크기 (값의 개수)'); ax2.set_ylabel('ETT 평균 MAE')
ax2.set_title('입력 패치 크기 Ablation
(ETT 데이터셋, 낮을수록 좋음)', pad=12)
ax2.grid(axis='y', alpha=0.4)
ax2.set_ylim(0.33, 0.46)
for bar, val in zip(bars, ett_mae):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
             f'{val}', ha='center', fontsize=10, fontweight='bold', color='#e6edf3')
ax2.text(1.0, 0.345, '← p=32 최적 균형점
    속도와 정확도', fontsize=9, color=GREEN,
         bbox=dict(boxstyle='round', facecolor=GREEN+'18', edgecolor=GREEN+'40'))

plt.tight_layout(pad=2)
plt.show()


## 8. 활용 분야 적합도 분석 <a id='8'></a>

TimesFM이 강한 분야 vs 사용하면 안 되는 분야를 정량화


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.patch.set_facecolor('#0d1117')

# ── 적합도 막대 ──────────────────────────────────────────
ax = axes[0]
ax.set_facecolor('#161b22')

use_cases = [
    ('웹/미디어 트래픽',     98, '사전학습 데이터 홈그라운드
(Wikipedia+Trends)'),
    ('리테일 수요 예측',     88, 'Rohlik 5,800 SKU:
LightGBM/TFT 능가'),
    ('교통량/이동 수단',     85, 'SF 자전거 720시간
GCP 공식 튜토리얼'),
    ('에너지 부하 예측',     82, '유럽 300가구 단기 전력
지도학습 PatchTST와 동등'),
    ('공급망 재고 계획',     80, '자동차 부품 MAE 15% 감소'),
    ('역학 곡선 (전염병)',   70, 'HFMD 예측 ARIMA 동급'),
    ('금융 수익률',          12, 'R²=-2.8%
CatBoost에게 패배'),
    ('초단기 시계열 (<30)',   5, '패치 크기 32 미달
→ shape 오류'),
]

labels_uc = [u[0] for u in use_cases]
scores_uc = [u[1] for u in use_cases]
descs     = [u[2] for u in use_cases]

bar_clrs = [GREEN if s >= 80 else ORANGE if s >= 60 else RED for s in scores_uc]
bars = ax.barh(labels_uc, scores_uc, color=bar_clrs, edgecolor='#30363d', height=0.6)
ax.axvline(80, color=GREEN,  linestyle=':', alpha=0.5, linewidth=1.2)
ax.axvline(60, color=ORANGE, linestyle=':', alpha=0.5, linewidth=1.2)
ax.set_xlabel('TimesFM 적합도 점수 (100 = 완벽)')
ax.set_title('산업별 TimesFM 적합도
(주관적 종합 평가)', pad=12)
ax.grid(axis='x', alpha=0.4)
ax.set_xlim(0, 115)

for bar, score in zip(bars, scores_uc):
    ax.text(score + 1.5, bar.get_y() + bar.get_height()/2,
            f'{score}%', va='center', fontsize=10, fontweight='bold', color='#e6edf3')

# 범례
legend_handles = [
    mpatches.Patch(color=GREEN,  label='강력 추천 (80%+)'),
    mpatches.Patch(color=ORANGE, label='조건부 추천 (60–79%)'),
    mpatches.Patch(color=RED,    label='사용 비추천 (<60%)'),
]
ax.legend(handles=legend_handles, loc='lower right', facecolor='#1c2128', edgecolor='#30363d')

# ── 상세 텍스트 ──────────────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor('#0d1117')
ax2.axis('off')
ax2.set_title('사례별 상세 설명', pad=12, color='#e6edf3')

y_pos = 0.97
for label, score, desc in use_cases:
    color = GREEN if score >= 80 else ORANGE if score >= 60 else RED
    icon  = '✅' if score >= 80 else '⚠️' if score >= 60 else '❌'
    ax2.text(0.02, y_pos, f'{icon} {label}', transform=ax2.transAxes,
             fontsize=11, fontweight='bold', color=color, va='top')
    ax2.text(0.05, y_pos - 0.035, desc, transform=ax2.transAxes,
             fontsize=9, color='#8b949e', va='top')
    y_pos -= 0.115

plt.tight_layout(pad=2)
plt.show()


## 9. 한계 레이더 차트 <a id='9'></a>

TimesFM의 알려진 한계를 정량화 (0 = 문제 없음, 10 = 매우 심각)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(16, 7), subplot_kw=dict(polar=True))
fig.patch.set_facecolor('#0d1117')

limitations = [
    '다변량\n지원 부족',
    '금융 데이터\n불일치',
    '체제 변환\n대응 불가',
    '짧은 시계열\n처리 한계',
    '설명\n불가능성',
    '확률 보정\n문제(v1/v2)',
    '벤치마크\n데이터 누출',
    '인프라\n복잡성',
]
scores_v1 = [8, 9, 7, 8, 8, 9, 7, 6]  # v1.0 기준
scores_v25 = [7, 9, 7, 7, 8, 3, 6, 5]  # v2.5 기준 (개선된 항목 반영)

N = len(limitations)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

for ax_idx, (ax, scores, title, color) in enumerate(zip(
    axes,
    [scores_v1, scores_v25],
    ['TimesFM v1.0 한계 프로파일', 'TimesFM v2.5 한계 프로파일
(개선된 항목 포함)'],
    [RED, ORANGE]
)):
    ax.set_facecolor('#161b22')
    vals = scores + scores[:1]

    ax.plot(angles, vals, 'o-', linewidth=2, color=color)
    ax.fill(angles, vals, alpha=0.25, color=color)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(limitations, size=9, color='#e6edf3')
    ax.set_ylim(0, 10)
    ax.set_yticks([2, 4, 6, 8, 10])
    ax.set_yticklabels(['2', '4', '6', '8', '10'], size=7, color='#8b949e')
    ax.grid(color='#30363d', alpha=0.5)
    ax.set_title(title, size=12, color='#e6edf3', pad=20, fontweight='bold')

    # 점수 표시
    for angle, val, label in zip(angles[:-1], scores, limitations):
        c = RED if val >= 8 else ORANGE if val >= 6 else GREEN
        ax.annotate(str(val), xy=(angle, val), fontsize=9, fontweight='bold', color=c, ha='center')

# 개선된 항목 설명
fig.text(0.5, 0.02,
         '🟢 v2.5에서 개선: 확률 보정 (9→3), 인프라 복잡성 (6→5), 짧은 시계열 (8→7)
'
         '🔴 근본적 한계: 금융 데이터 불일치, 다변량 지원 부족은 아키텍처 제약으로 지속',
         ha='center', fontsize=10, color='#8b949e',
         bbox=dict(boxstyle='round', facecolor='#161b22', edgecolor='#30363d'))

plt.tight_layout(pad=2)
plt.subplots_adjust(bottom=0.12)
plt.show()


## 10. 모델 선택 가이드 요약 <a id='10'></a>

2026년 기준, 상황별 최적 모델을 한눈에 정리합니다.


In [ ]:
fig, ax = plt.subplots(figsize=(16, 9))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#0d1117')
ax.axis('off')

guide = [
    ('상황',                         '추천 모델',                    '이유',                                  BLUE),
    ('빠른 Zero-shot + SQL 환경',    'TimesFM 2.5 (BigQuery)',      '설치 불필요, SQL 한 줄로 예측',          GREEN),
    ('확률 예측이 핵심',              'Chronos-2 (Amazon)',           'GIFT-Eval 비누출 1위, 보정된 분위수',    '#ff9900'),
    ('다변량 교차 시계열',            'Moirai-2 / TiDE',              '시리즈 간 상호작용 학습 가능',           '#00a1e0'),
    ('금융 수익/변동성',              'GARCH + CatBoost/LGB',         'TimesFM 분포 불일치 근본 문제',          RED),
    ('추세/계절성 분해 필요',         'Prophet / ARIMA_PLUS',         '컴포넌트 분해 + 비즈니스 설명 가능',     ORANGE),
    ('데이터 30개 미만',             'ETS / 지수평활',               '패치 크기 32 미달 → 사용 불가',          RED),
    ('경량 엣지 환경',               'TinyTimeMixer (IBM, 5M)',      '40배 경량, 벤치마크 경쟁력',             CYAN),
    ('도메인 특화 + 데이터 충분',    'TimesFM Fine-tuning (LoRA)',   '1% 파라미터만 학습, 큰 성능 향상',       PURPLE),
]

col_widths = [0.30, 0.28, 0.38]
col_x      = [0.01, 0.32, 0.61]
row_h      = 0.088
start_y    = 0.96

for row_idx, row in enumerate(guide):
    y = start_y - row_idx * row_h
    is_header = row_idx == 0

    # 배경
    bg_color = '#1c2128' if is_header else ('#161b22' if row_idx % 2 == 0 else '#0d1117')
    rect = mpatches.FancyBboxPatch((0, y - row_h * 0.8), 0.99, row_h * 0.85,
                                    boxstyle="round,pad=0.005",
                                    facecolor=bg_color,
                                    edgecolor='#30363d' if not is_header else row[3],
                                    linewidth=1.5 if is_header else 0.5,
                                    transform=ax.transAxes)
    ax.add_patch(rect)

    for col_idx, (text, cx, cw) in enumerate(zip(row[:3], col_x, col_widths)):
        color = row[3] if (col_idx == 1 and not is_header) else ('#e6edf3' if is_header else '#c9d1d9')
        fw = 'bold' if (is_header or col_idx == 1) else 'normal'
        fs = 11 if is_header else 10
        ax.text(cx + 0.01, y - row_h * 0.35, text,
                transform=ax.transAxes, fontsize=fs, fontweight=fw,
                color=color, va='center', wrap=True)

ax.set_title('TimesFM & 경쟁 모델 상황별 선택 가이드 (2026 기준)',
             fontsize=14, fontweight='bold', color='#e6edf3', pad=20)

plt.tight_layout()
plt.show()

print()
print("=" * 65)
print("📌 핵심 결론")
print("=" * 65)
print("✅ TimesFM은 '웹 트래픽·리테일·에너지' 등 일반 업무 시계열에서")
print("   Zero-shot으로 강력한 기준점을 제공합니다.")
print()
print("✅ BigQuery ML의 AI.FORECAST로 SQL 한 줄이면 충분합니다.")
print()
print("⚠️  2026년 리더보드 1위는 Chronos-2(Amazon)로 변경되었으며,")
print("   TimesFM은 Top 3 수준입니다.")
print()
print("❌ 금융 수익률, 30개 미만 데이터, 다변량 교차 예측에는")
print("   TimesFM 대신 도메인 특화 모델을 사용하세요.")
print("=" * 65)
